In [1]:
from torchcfm.optimal_transport import OTPlanSampler
from torchcfm.conditional_flow_matching import ConditionalFlowMatcher
from torchcfm.utils import eight_normal_sample
import torch
import math
import numpy as np

In [ ]:
k = float(8.0)
c = float(5.0)
t = torch.tensor(0.5)
eps = 1e-8
k = torch.as_tensor(k, dtype=t.dtype, device=t.device)
c = torch.as_tensor(c, dtype=t.dtype, device=t.device)
left = torch.sigmoid(-k * c)
right = torch.sigmoid(k * (1.0 - c))
denom = torch.clamp(right - left, min=eps,)
denom

tensor(1.0000e-08)

In [8]:
ot = OTPlanSampler(method="exact")
cfm = ConditionalFlowMatcher(sigma = 1.0)
dim = 2
var = 1
scale = 1
n = 10
shifted_center = 2
m = torch.distributions.multivariate_normal.MultivariateNormal(
    torch.zeros(dim), math.sqrt(var) * torch.eye(dim)
)

x0 = m.sample((n,))
x1 = m.sample((n,)) + shifted_center

In [11]:
from typing import Union
def pad_t_like_x(t, x) -> Union[torch.Tensor, float, int]:
    """Function to reshape the time vector t by the number of dimensions of x.

    Parameters
    ----------
    x : Tensor, shape (bs, *dim)
        represents the source minibatch
    t : FloatTensor, shape (bs)

    Returns
    -------
    t : Tensor, shape (bs, number of x dimensions)

    Example
    -------
    x: Tensor (bs, C, W, H)
    t: Vector (bs)
    pad_t_like_x(t, x): Tensor (bs, 1, 1, 1)
    """
    if isinstance(t, (float, int)):
        return t
    return t.reshape(-1, *([1] * (x.dim() - 1)))

pad_t_like_x(t=0.5, x = x0)

0.5

In [14]:
ot = OTPlanSampler(method="exact")

In [15]:
ot.sample_plan(x0, x1)

(tensor([[ 0.8601,  0.6738],
         [-0.1058, -1.2377],
         [-1.6288,  0.3073],
         [-0.9685, -1.0524],
         [-0.0657, -0.1155],
         [ 0.8601,  0.6738],
         [-0.1058, -1.2377],
         [-0.9685, -1.0524],
         [ 0.8601,  0.6738],
         [-2.7119, -0.7185]]),
 tensor([[3.3054, 2.6827],
         [3.6508, 0.5224],
         [1.4343, 2.3204],
         [1.7531, 0.8527],
         [2.3821, 2.0078],
         [3.3054, 2.6827],
         [3.6508, 0.5224],
         [1.7531, 0.8527],
         [3.3054, 2.6827],
         [0.6725, 1.4261]]))

In [12]:
cfm

tensor([[-0.1058, -1.2377],
        [-0.7567,  0.7094],
        [-0.0657, -0.1155],
        [-3.0878,  1.3890],
        [-0.9685, -1.0524],
        [-0.2290, -1.8113],
        [-0.7980, -1.1411],
        [-2.7119, -0.7185],
        [ 0.8601,  0.6738],
        [-1.6288,  0.3073]])

In [13]:
cfm.compute_mu_t(x0, x1, t=0.5)

tensor([[ 0.9448, -0.3434],
        [ 0.6883,  0.9366],
        [ 1.6198,  1.2836],
        [-0.3528,  1.6984],
        [-0.1480,  0.1869],
        [ 0.6026,  0.2546],
        [ 1.0631,  0.8179],
        [ 0.4694, -0.0981],
        [ 1.4076,  2.2236],
        [ 0.0622,  0.5800]])